In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config
from lib_etl.utility import *

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
def calculate_gas_spend_by_club(detail):
    """
    Calculates the total gas spend at each club.

    Args:
        detail (job.spark.sql.Dataframe): the detail table

    Returns:
        gas_spend (job.spark.sql.Dataframe): table with last four week gas spend per club
    """
    detail = (
        detail.filter(
            (detail.SALES_CTGRY_CD == "01") & (detail.EXTENDED_PRC_AMT > 0.0)
        )
        .groupby("SITE_NBR", "FISCAL_WEEK_END")
        .agg(f.sum(detail.EXTENDED_PRC_AMT).alias("GAS_SPEND"))
    )

    window = create_window("SITE_NBR", 4)

    gas_spend = detail.withColumn(
        "EPOCH", to_epoch(detail.FISCAL_WEEK_END)
    ).withColumn("L4W_GAS_SPEND", f.sum("GAS_SPEND").over(window))

    return gas_spend


def club_gas_start_week(gas_spend):
    """
    Returns a two column with SITE_NBR and the first FISCAL_WEEK_END
    that it had gas.

    Args:
        gas_spend (job.spark.sql.Dataframe): table with last four week gas spend per club

    Returns:
        first_week_has_gas (job.spark.sql.Dataframe): Dataframe with club and gas start week
    """
    last_four_week_spend_threshold = 10000
    gas_spend = gas_spend.filter(
        gas_spend.L4W_GAS_SPEND >= last_four_week_spend_threshold
    )
    first_week_has_gas = gas_spend.groupby("SITE_NBR").agg(
        f.min("FISCAL_WEEK_END").alias("FIRST_FW_HAS_GAS")
    )
    return first_week_has_gas


def add_gas_start_date(club, detail):
    """
    Adds gas start date to club intermediate

    Args:
        club (job.spark.sql.Dataframe): the club intermediate
        detail (job.spark.sql.Dataframe): the detail intermediate
    """
    gas_spend = calculate_gas_spend_by_club(detail)

    first_week_has_gas = club_gas_start_week(gas_spend)

    club = club.join(first_week_has_gas, ["SITE_NBR"], "left_outer")
    return club


In [0]:
club = spark.table(bronze_master_club_with_brand)

detail = spark.table(silver_transaction_fiscal_detail)

club.createOrReplaceTempView("club")

club = spark.sql("""
    select
        cast(_c0 as integer) as SITE_NBR
        ,cast(_c1 as string) as SITE_NAME_2
        ,cast(_c2 as string) as ADDR_LINE_2
        ,cast(_c3 as string) as CITY_NAME
        ,cast(_c4 as string) as STATE_CD
        ,cast(_c5 as string) as ZIP_CD
        ,cast(_c6 as integer) as ZN_NBR
        ,cast(_c7 as integer) as RGN_NBR
        ,cast(_c8 as string) as SITE_TYPE
        ,cast(_c9 as string) as COMP_STTS
    from club
""")

# If the bronze table schema is updated to proper column names in the future, this is the code to be run:
# club = spark.sql(""" 
#     select
#         cast(SITE_NBR       as integer) as SITE_NBR
#         ,cast(SITE_NAME_2   as string)  as SITE_NAME_2
#         ,cast(ADDR_LINE_2   as string)  as ADDR_LINE_2
#         ,cast(CITY_NAME     as string)  as CITY_NAME
#         ,cast(STATE_CD      as string)  as STATE_CD
#         ,cast(ZIP_CD        as string)  as ZIP_CD
#         ,cast(ZN_NBR        as integer) as ZN_NBR
#         ,cast(RGN_NBR       as integer) as RGN_NBR
#         ,cast(SITE_TYPE     as string)  as SITE_TYPE
#         ,cast(COMP_STTS     as string)  as COMP_STTS
#     from club
# """)

df_club = add_gas_start_date(club, detail)
df_club.dropDuplicates()

df_club.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'club', config_validation, df_club, stats_etl_path
    )

### Merge

In [0]:
df_club.write.mode("overwrite").saveAsTable(silver_master_club_with_brand)

if archive_flag:
    save_archive(df_club, silver_master_club_with_brand_archive, run_as_date)